In [0]:
%pip install --upgrade youtube-transcript-api google-genai
dbutils.library.restartPython()

In [0]:
%pip install google-genai youtube-transcript-api google-api-python-client
dbutils.library.restartPython()

In [0]:
import os
import json
import time
import base64
from google import genai
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi

GEMINI_API_KEY = dbutils.secrets.get(scope="courseify", key="gemini-api")
YOUTUBE_API_KEY = dbutils.secrets.get(scope="courseify", key="youtube-api")
PLAYLIST_ID = "PLdo5W4Nhv31bZSiqiOL5ta39vSnBxpOPT"

client = genai.Client(api_key=GEMINI_API_KEY)
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

# (Keep your SYLLABUS, get_playlist_videos, search_external_video, and map_videos_to_topics functions here just as they were)

def get_transcript(video_id):
    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        return " ".join([item['text'] for item in transcript_list])
    except Exception:
        try:
            transcripts = YouTubeTranscriptApi.list_transcripts(video_id)
            for t in transcripts: return " ".join([item['text'] for item in t.fetch()])
        except Exception: return ""

def generate_master_notes(combined_transcript, topic_title):
    prompt = f"""
    You are an Elite Coding Instructor. Generate an extremely comprehensive study guide and practice test for the Core Topic: '{topic_title}'.

    You MUST return EXACTLY a JSON object matching this schema:
    {{
        "markdown_notes": "A to Z Markdown notes including Overview, Core Concepts, Code Examples, Pitfalls, and Interview Questions.",
        "practice_test": [
            {{
                "question": "The question text",
                "options": ["A", "B", "C", "D"],
                "answer_index": 0
            }}
        ]
    }}
    Ensure exactly 5 questions in the practice test.
    
    Source Transcript: {combined_transcript[:25000]}
    """
    try:
        # THE FIX: Native JSON Enforcement
        response = client.models.generate_content(
            model='gemini-2.5-flash', 
            contents=prompt, 
            config={
                'temperature': 0.3,
                'response_mime_type': 'application/json' # Guarantees valid JSON
            }
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"Gemini Error: {e}")
        return None

def build_course():
    videos = get_playlist_videos()
    mapping_data = map_videos_to_topics(videos)
    
    # Organize structure
    course_structure = {s['id']: {"title": s['title'], "videos": []} for s in SYLLABUS}
    video_details = {v['video_id']: {"original_title": v['title'], "channel_name": v['channel_name']} for v in videos}
    
    for m in mapping_data:
        topic_id = m['topic_id']
        v_id = m['video_id']
        if topic_id in course_structure and v_id in video_details:
            course_structure[topic_id]['videos'].append({
                "video_id": v_id,
                "refined_title": m.get('refined_title', video_details[v_id]['original_title']),
            })
            
    final_output = {"course_notes": {"py_notes_shared": {}}}
    
    for topic_id, data in course_structure.items():
        print(f"\n⏳ Processing Topic: {data['title']}")
        
        # To save quota, skip topics you already generated today!
        if str(topic_id) in ["0", "1", "3", "5", "6", "12", "24"]:
            print("   ⏩ Skipping (Already Generated)")
            continue
            
        if not data['videos']: 
            ext_video = search_external_video(data['title'])
            if ext_video: data['videos'].append(ext_video)
            else: continue
                
        combined_transcript = ""
        for v in data['videos']:
            t = get_transcript(v['video_id'])
            if t: combined_transcript += f"\n{t}"
            
        if not combined_transcript:
            combined_transcript = f"Generate comprehensive notes for {data['title']} in Python."
            
        ai_data = generate_master_notes(combined_transcript, data['title'])
        
        if ai_data:
            # THE FIX: Base64 Encoding immediately upon generation
            raw_markdown = ai_data.get('markdown_notes', '')
            b64_notes = base64.b64encode(raw_markdown.encode('utf-8')).decode('utf-8')
            
            final_output["course_notes"]["py_notes_shared"][str(topic_id)] = {
                "notes_base64": b64_notes,
                "practice_test": ai_data.get('practice_test', [])
            }
            print("   ✅ Processed Successfully in Target Format!")
            
        print("   💤 Sleeping for 60 seconds to respect API Rate Limits...")
        time.sleep(60) # Slows down requests to help prevent 429 errors

    with open("purified_course_data.json", "w") as f:
        json.dump(final_output, f, indent=4)
        
    print("\n🚀 DONE!")

if __name__ == "__main__":
    build_course()

In [0]:
import os
import json
import time
from google import genai
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi

# ==========================================
# 1. SETUP & API KEYS (Fixed Syntax)
# ==========================================
# Make sure your secret scopes and keys exactly match what is in Databricks
GEMINI_API_KEY = dbutils.secrets.get(scope="courseify", key="gemini-api")
YOUTUBE_API_KEY = dbutils.secrets.get(scope="courseify", key="youtube-api")
PLAYLIST_ID = "PLdo5W4Nhv31bZSiqiOL5ta39vSnBxpOPT" # Jenny's Python

client = genai.Client(api_key=GEMINI_API_KEY)
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

# ==========================================
# 2. MANA CORE SYLLABUS (25 TOPICS)
# ==========================================
SYLLABUS = [
    {"id": 0, "title": "Introduction & Setup"}, {"id": 1, "title": "Variables & Data Types"},
    {"id": 2, "title": "Type Casting"}, {"id": 3, "title": "Basic I/O"},
    {"id": 4, "title": "Operators"}, {"id": 5, "title": "Conditional Statements"},
    {"id": 6, "title": "Loops & Loop Control"}, {"id": 7, "title": "Core Data Structures (Lists, Tuples, Sets, Dictionaries)"},
    {"id": 8, "title": "Comprehensions"}, {"id": 9, "title": "Functions Basics & Arguments"},
    {"id": 10, "title": "Scope (Local vs Global)"}, {"id": 11, "title": "Lambda Functions"},
    {"id": 12, "title": "Built-in Functions"}, {"id": 13, "title": "Modules & Packages"},
    {"id": 14, "title": "Classes & Objects"}, {"id": 15, "title": "Attributes & Methods"},
    {"id": 16, "title": "Inheritance & Polymorphism"}, {"id": 17, "title": "Encapsulation & Abstraction"},
    {"id": 18, "title": "Magic/Dunder Methods"}, {"id": 19, "title": "File Handling"},
    {"id": 20, "title": "Exception Handling"}, {"id": 21, "title": "Iterators & Generators"},
    {"id": 22, "title": "Decorators"}, {"id": 23, "title": "Regular Expressions (Regex)"},
    {"id": 24, "title": "Advanced Concepts Overview"}
]

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def get_playlist_videos():
    print("Fetching videos from YouTube Playlist...")
    videos = []
    next_token = None
    while True:
        req = youtube.playlistItems().list(part="snippet", playlistId=PLAYLIST_ID, maxResults=50, pageToken=next_token)
        res = req.execute()
        for item in res['items']:
            title = item['snippet']['title']
            if title not in ["Private video", "Deleted video"]:
                videos.append({
                    "video_id": item['snippet']['resourceId']['videoId'],
                    "title": title,
                    "channel_name": item['snippet']['videoOwnerChannelTitle'] # Added channel name
                })
        next_token = res.get('nextPageToken')
        if not next_token: break
    return videos

def search_external_video(topic_title):
    """Fallback: Mana list lo leni topic ki YouTube nundi oka best video techukundam."""
    print(f"   🔍 Searching external YouTube video for missing topic: {topic_title}...")
    try:
        req = youtube.search().list(
            q=f"Python {topic_title} tutorial for beginners",
            part="snippet",
            type="video",
            maxResults=1
        )
        res = req.execute()
        if res['items']:
            item = res['items'][0]
            return {
                "video_id": item['id']['videoId'],
                "title": item['snippet']['title'],
                "channel_name": item['snippet']['channelTitle'],
                "refined_title": f"Master {topic_title}" # Default clean title
            }
    except Exception as e:
        print(f"   ❌ YouTube Search failed: {e}")
    return None

def get_transcript(video_id):
    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        return " ".join([item['text'] for item in transcript_list])
    except Exception:
        try:
            transcripts = YouTubeTranscriptApi.list_transcripts(video_id)
            for t in transcripts: return " ".join([item['text'] for item in t.fetch()])
        except Exception: return ""

# 🔥 AI MAPPER: Matches videos to topics AND generates better names
def map_videos_to_topics(videos):
    print("🧠 Asking AI to map videos to syllabus topics and assign smart names...")
    
    syllabus_str = "\n".join([f"ID {s['id']}: {s['title']}" for s in SYLLABUS])
    videos_str = "\n".join([f"VID {v['video_id']} : {v['title']}" for v in videos])
    
    prompt = f"""
    You are an expert curriculum designer. Map the following YouTube videos to the most relevant Syllabus Topic.
    If multiple videos belong to the same topic, group them under the same topic_id, but provide a logical `refined_title` for each video (e.g., if there are 3 videos for Variables, name them "Variables Part 1: Basics", "Variables Part 2: Data Types", etc. or base it on the original title's context).
    
    SYLLABUS TOPICS:
    {syllabus_str}
    
    VIDEOS:
    {videos_str}
    
    Return EXACTLY a JSON array of objects. Format: 
    [{{
        "video_id": "...", 
        "topic_id": 0,
        "refined_title": "Clean, descriptive name for this specific video part"
    }}, ...]
    """
    
    response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt, config={'temperature': 0.1})
    res_text = response.text.replace('```json\n', '').replace('```', '').strip()
    return json.loads(res_text)

# 🔥 AI CONTENT GENERATOR (Super Detailed A to Z)
def generate_master_notes(combined_transcript, topic_title):
    prompt = f"""
    You are an Elite Coding Instructor. Generate an extremely comprehensive, 'Topper-Level' study guide and practice test for the Core Topic: '{topic_title}'.
    This needs to go from Absolute Basics to Advanced levels. Cover every single point A to Z. Explain it so clearly that a beginner can understand, but include depth for advanced learners.

    OUTPUT FORMAT: EXACTLY a valid JSON object.
    
    1. "markdown_notes": Beautiful markdown text. MUST Include:
       - 🎯 **Overview & Intuition:** Explain the concept in simple layman's terms with a real-life analogy.
       - 📌 **Core Concepts (A to Z):** Detailed subheadings covering everything. Break down definitions, syntax, and rules.
       - 💻 **Code in Action:** At least 3-4 practical, step-by-step code blocks with very detailed inline comments explaining WHAT and WHY.
       - 💡 **Pro-Tips & Tricks:** Shortcuts, best practices, and industry standards.
       - ⚠️ **Common Pitfalls & Mistakes:** What do beginners usually do wrong and how to avoid it.
       - 🎙️ **Important Interview Questions:** 3 frequently asked interview questions on this topic with answers.
       - 📝 **Summary Note Points:** Quick bullet points for final revision.
       
    2. "practice_test": Array of EXACTLY 5 challenging multiple-choice questions covering the entire topic.
       Format: {{"q": "?", "opts": ["A", "B", "C", "D"], "ans": 0, "exp": "Explanation"}}
    
    Source Transcript: {combined_transcript[:25000]}
    """
    try:
        response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt, config={'temperature': 0.3})
        res_text = response.text.replace('```json\n', '').replace('```', '').strip()
        return json.loads(res_text)
    except Exception as e:
        print(f"Gemini Error: {e}")
        return None

# ==========================================
# 4. MAIN EXECUTION FLOW
# ==========================================
def build_course():
    videos = get_playlist_videos()
    mapping_data = map_videos_to_topics(videos)
    
    # Organize videos under our 25 Topics
    course_structure = {s['id']: {"title": s['title'], "videos": []} for s in SYLLABUS}
    
    # Create lookup dictionaries
    video_details = {v['video_id']: {"original_title": v['title'], "channel_name": v['channel_name']} for v in videos}
    
    for m in mapping_data:
        topic_id = m['topic_id']
        v_id = m['video_id']
        if topic_id in course_structure and v_id in video_details:
            course_structure[topic_id]['videos'].append({
                "video_id": v_id,
                "original_title": video_details[v_id]['original_title'],
                "refined_title": m.get('refined_title', video_details[v_id]['original_title']),
                "channel_name": video_details[v_id]['channel_name']
            })
            
    final_output = {"course_notes": {"py_notes_shared": {}}}
    
    for topic_id, data in course_structure.items():
        print(f"\n⏳ Processing Master Topic: {data['title']} ({len(data['videos'])} videos)")
        
        # 🟢 FALLBACK LOGIC: If no videos from playlist, fetch external video
        if not data['videos']: 
            print(f"   ⚠️ No videos found in playlist for '{data['title']}'.")
            ext_video = search_external_video(data['title'])
            if ext_video:
                data['videos'].append(ext_video)
            else:
                print("   ⏭️ Skipping as no external video could be fetched.")
                continue
                
        # Combine all transcripts for this topic
        combined_transcript = ""
        for v in data['videos']:
            print(f"   -> Fetching Transcript for: {v['refined_title']} (Channel: {v['channel_name']})")
            t = get_transcript(v['video_id'])
            if t: 
                combined_transcript += f"\n\n--- Content from {v['refined_title']} ---\n" + t
            
        if not combined_transcript:
            print("   ⚠️ No transcripts available. Generating general notes based on AI knowledge...")
            combined_transcript = f"Generate comprehensive notes for {data['title']} in Python."
            
        print("   🧠 Generating Super Detailed A-to-Z Master Notes & Quiz...")
        ai_data = generate_master_notes(combined_transcript, data['title'])
        
        if ai_data:
            final_output["course_notes"]["py_notes_shared"][str(topic_id)] = {
                "topic_name": data['title'],
                "notes": ai_data['markdown_notes'],
                "practice_test": ai_data['practice_test'],
                "sub_videos": data['videos'] # This contains video_id, refined_title, channel_name!
            }
            print("   ✅ Master Concept Processed Successfully!")
            
        time.sleep(5) # API limit safety

    with open("purified_course_data.json", "w") as f:
        json.dump(final_output, f, indent=4)
        
    print("\n🚀 DONE! Check 'purified_course_data.json'.")

if __name__ == "__main__":
    build_course()

In [0]:
import os
import json
import time
from google import genai
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi

# ==========================================
# 1. SETUP & API KEYS
# ==========================================
GEMINI_API_KEY = dbutils.secrets.get("courseify", "key"="gemini-api")
YOUTUBE_API_KEY = dbutils.secrets.get("courseify", "youtube-api")
PLAYLIST_ID = "PLdo5W4Nhv31bZSiqiOL5ta39vSnBxpOPT" # Jenny's Python

client = genai.Client(api_key=GEMINI_API_KEY)
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

# ==========================================
# 2. MANA CORE SYLLABUS (25 TOPICS)
# ==========================================
SYLLABUS = [
    {"id": 0, "title": "Introduction & Setup"}, {"id": 1, "title": "Variables & Data Types"},
    {"id": 2, "title": "Type Casting"}, {"id": 3, "title": "Basic I/O"},
    {"id": 4, "title": "Operators"}, {"id": 5, "title": "Conditional Statements"},
    {"id": 6, "title": "Loops & Loop Control"}, {"id": 7, "title": "Core Data Structures (Lists, Tuples, Sets, Dictionaries)"},
    {"id": 8, "title": "Comprehensions"}, {"id": 9, "title": "Functions Basics & Arguments"},
    {"id": 10, "title": "Scope (Local vs Global)"}, {"id": 11, "title": "Lambda Functions"},
    {"id": 12, "title": "Built-in Functions"}, {"id": 13, "title": "Modules & Packages"},
    {"id": 14, "title": "Classes & Objects"}, {"id": 15, "title": "Attributes & Methods"},
    {"id": 16, "title": "Inheritance & Polymorphism"}, {"id": 17, "title": "Encapsulation & Abstraction"},
    {"id": 18, "title": "Magic/Dunder Methods"}, {"id": 19, "title": "File Handling"},
    {"id": 20, "title": "Exception Handling"}, {"id": 21, "title": "Iterators & Generators"},
    {"id": 22, "title": "Decorators"}, {"id": 23, "title": "Regular Expressions (Regex)"},
    {"id": 24, "title": "Advanced Concepts Overview"}
]

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def get_playlist_videos():
    print("Fetching videos from YouTube Playlist...")
    videos = []
    next_token = None
    while True:
        req = youtube.playlistItems().list(part="snippet", playlistId=PLAYLIST_ID, maxResults=50, pageToken=next_token)
        res = req.execute()
        for item in res['items']:
            title = item['snippet']['title']
            if title not in ["Private video", "Deleted video"]:
                videos.append({
                    "video_id": item['snippet']['resourceId']['videoId'],
                    "title": title
                })
        next_token = res.get('nextPageToken')
        if not next_token: break
    return videos

def get_transcript(video_id):
    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        return " ".join([item['text'] for item in transcript_list])
    except Exception:
        try:
            transcripts = YouTubeTranscriptApi.list_transcripts(video_id)
            for t in transcripts: return " ".join([item['text'] for item in t.fetch()])
        except Exception: return ""

# 🔥 AI MAPPER: Matches videos to our 25 Topics
def map_videos_to_topics(videos):
    print("🧠 Asking AI to map videos to syllabus topics...")
    
    syllabus_str = "\n".join([f"ID {s['id']}: {s['title']}" for s in SYLLABUS])
    videos_str = "\n".join([f"VID {v['video_id']} : {v['title']}" for v in videos])
    
    prompt = f"""
    You are an expert curriculum designer. Map the following YouTube videos to the most relevant Syllabus Topic.
    
    SYLLABUS TOPICS:
    {syllabus_str}
    
    VIDEOS:
    {videos_str}
    
    Return EXACTLY a JSON array of objects. Format: [{{"video_id": "...", "topic_id": 0}}, ...]
    Assign every video to exactly one topic_id (0-24).
    """
    
    response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt, config={'temperature': 0.1})
    res_text = response.text.replace('```json\n', '').replace('```', '').strip()
    return json.loads(res_text)

# 🔥 AI CONTENT GENERATOR (Consolidated per Topic)
def generate_master_notes(combined_transcript, topic_title):
    prompt = f"""
    You are an Elite Coding Instructor. Generate a 'Topper-Level' study guide and practice test for the Core Topic: '{topic_title}'.
    This covers multiple videos, so consolidate the knowledge.
    
    OUTPUT FORMAT: EXACTLY a valid JSON object.
    
    1. "markdown_notes": Beautiful markdown text. Include:
       - 🎯 **Overview**
       - 📌 **Core Concepts** (Subheadings for different sub-topics)
       - 💻 **Code in Action** (2-3 practical code blocks with comments)
       - 💡 **Pro-Tip / ⚠️ Common Pitfall**
    2. "practice_test": Array of EXACTLY 5 multiple-choice questions covering the entire topic.
       Format: {{"q": "?", "opts": ["A", "B", "C", "D"], "ans": 0, "exp": "Explanation"}}
    
    Source Transcript: {combined_transcript[:25000]}
    """
    try:
        response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt, config={'temperature': 0.2})
        res_text = response.text.replace('```json\n', '').replace('```', '').strip()
        return json.loads(res_text)
    except Exception as e:
        print(f"Gemini Error: {e}")
        return None

# ==========================================
# 4. MAIN EXECUTION FLOW
# ==========================================
def build_course():
    # 1. Fetch & Map
    videos = get_playlist_videos()
    mapping_data = map_videos_to_topics(videos)
    
    # Organize videos under our 25 Topics
    course_structure = {s['id']: {"title": s['title'], "videos": [], "notes_base64": "", "practice_test": []} for s in SYLLABUS}
    
    video_dict = {v['video_id']: v['title'] for v in videos}
    for m in mapping_data:
        topic_id = m['topic_id']
        if topic_id in course_structure:
            course_structure[topic_id]['videos'].append({
                "video_id": m['video_id'],
                "title": video_dict.get(m['video_id'], "")
            })
            
    # 2. Process Each of the 25 Topics
    final_output = {"course_notes": {"py_notes_shared": {}}}
    
    for topic_id, data in course_structure.items():
        if not data['videos']: 
            continue # Skip if no videos matched
            
        print(f"\n⏳ Processing Master Topic: {data['title']} ({len(data['videos'])} videos)")
        
        # Combine all transcripts for this topic
        combined_transcript = ""
        for v in data['videos']:
            print(f"   -> Fetching: {v['title']}")
            t = get_transcript(v['video_id'])
            if t: combined_transcript += f"\n\n--- Content from {v['title']} ---\n" + t
            
        if not combined_transcript:
            print("   ⚠️ No transcripts available for this topic.")
            continue
            
        # Generate 1 Master Note for all videos combined
        print("   🧠 Generating Master Notes & Quiz...")
        ai_data = generate_master_notes(combined_transcript, data['title'])
        
        if ai_data:
            # Saving to final JSON structure
            final_output["course_notes"]["py_notes_shared"][str(topic_id)] = {
                "notes": ai_data['markdown_notes'],
                "practice_test": ai_data['practice_test'],
                "sub_videos": data['videos'] # Keeps track of which videos belong here
            }
            print("   ✅ Master Concept Processed!")
            
        time.sleep(5) # API limit safety

    # 3. Export exactly matching your requirement!
    with open("purified_course_data.json", "w") as f:
        json.dump(final_output, f, indent=4)
        
    print("\n🚀 DONE! Check 'purified_course_data.json'.")

if __name__ == "__main__":
    build_course()

In [0]:
# pip uninstall youtube-transcript-api -y
%pip install youtube-transcript-api==0.6.2


In [0]:
dbutils.library.restartPython()

In [0]:
from youtube_transcript_api import YouTubeTranscriptApi

print("Fetching transcript...")
transcript = YouTubeTranscriptApi.get_transcript("DInMru2Eq6E") # Jenny's 1st video ID
print(transcript[:500]) # First 500 characters print chestundi
print("\n✅ SUCCESS! Transcript is working locally.")

In [0]:
import os
import json
import time
from google import genai
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi

# ==========================================
# 1. SETUP YOUR API KEYS HERE
# ==========================================
GEMINI_API_KEY = "MEE_GEMINI_API_KEY_IKKADA_PETTU"
YOUTUBE_API_KEY = "MEE_YOUTUBE_API_KEY_IKKADA_PETTU"

PLAYLIST_ID = "PLdo5W4Nhv31bZSiqiOL5ta39vSnBxpOPT" # Jenny's Python
LOCAL_DB_FILE = "course_local_db.json"
FINAL_OUTPUT_FILE = "py_notes_final.json"

client = genai.Client(api_key=GEMINI_API_KEY)
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

# ==========================================
# 2. FUNCTIONS
# ==========================================
def get_playlist_videos():
    print("Fetching Playlist from YouTube...")
    videos = []
    next_token = None
    topic_index = 0
    while True:
        req = youtube.playlistItems().list(part="snippet", playlistId=PLAYLIST_ID, maxResults=50, pageToken=next_token)
        res = req.execute()
        for item in res['items']:
            title = item['snippet']['title']
            if title not in ["Private video", "Deleted video"]:
                videos.append({
                    "topic_id": f"topic_{topic_index}",
                    "video_id": item['snippet']['resourceId']['videoId'],
                    "title": title,
                    "is_processed": False,
                    "ai_data": None
                })
                topic_index += 1
        next_token = res.get('nextPageToken')
        if not next_token: break
    return videos

# 🔥 BULLET-PROOF TRANSCRIPT EXTRACTOR 🔥
def get_transcript(video_id):
    try:
        # Attempt 1: Fetch default English transcript
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        return " ".join([item['text'] for item in transcript_list])
    except Exception as e1:
        try:
            # Attempt 2: If default fails, fetch ANY available transcript (Auto-generated/Hindi/etc)
            transcripts = YouTubeTranscriptApi.list_transcripts(video_id)
            for t in transcripts:
                # Grab the very first transcript available and return it
                return " ".join([item['text'] for item in t.fetch()])
        except Exception as e2:
            print(f"      ⚠️ Reason 1: {e1}")
            print(f"      ⚠️ Reason 2: {e2}")
            return None
        
def generate_ai_content(transcript, title):
    prompt = f"""
    You are an Elite Silicon Valley Coding Instructor and a Master UX Content Designer for a premium educational app.
    Your objective is to transform the raw video transcript into a world-class, highly engaging, 'Topper-Level' study guide and adaptive practice test.
    
    Topic Context: '{title}'
    
    OUTPUT FORMAT: 
    Return EXACTLY a valid JSON object. Do not include ```json blocks or any plain text outside the JSON.
    
    JSON STRUCTURE & STRICT REQUIREMENTS:
    
    1. "suggested_name" (String): 
       - A clean, concise, professional title. 
       - Remove all YouTube fluff, channel names, pipe characters (|), and lecture numbers. 
       - Example: "History of Python | Python Tutorials for Beginners #lec2" -> "History of Python".
       
    2. "markdown_notes" (String):
       - Create visually stunning, easy-to-scan, and highly educational markdown text.
       - Structure it perfectly using this flow:
         * 🎯 **Overview:** A crisp 2-line summary of what this topic covers.
         * 📌 **Core Concepts:** Use clear sub-headings (###) and bullet points. Bold important keywords.
         * 💻 **Code in Action:** Provide 1-2 highly practical Python code examples. You MUST include detailed, easy-to-understand inline comments (`#`) explaining the logic step-by-step.
         * 💡 **Pro-Tip / ⚠️ Common Pitfall:** Add a small practical tip or a warning about a common mistake students make.
       - EXERCISE RULE: If the transcript is a "Coding Exercise", ignore the standard flow and structure the notes as: "Problem Statement" -> "Logic & Approach" -> "Solution Code" -> "Code Walkthrough".
       
    3. "practice_test" (Array of Objects):
       - Provide EXACTLY 3 multiple-choice questions.
       - ADAPTIVE RULE: 
         - If theory-based: Ask conceptual questions to test understanding.
         - If code/exercise-based: Ask logic-based questions or "Predict the Output" questions (using small code snippets in the question text).
       - Object Format: {{"q": "Question text?", "opts": ["Option 1", "Option 2", "Option 3", "Option 4"], "ans": 0, "exp": "One-line clear explanation why this is correct."}}
       
    Transcript to process: 
    {transcript[:8000]}
    """
    try:
        response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt, config={'temperature': 0.2})
        res_text = response.text.replace('```json\n', '').replace('```', '').strip()
        return json.loads(res_text)
    except Exception as e:
        print(f"Gemini Error: {e}")
        return None
# ==========================================
# 3. THE MAIN ENGINE
# ==========================================
def run_local_engine():
    if os.path.exists(LOCAL_DB_FILE):
        with open(LOCAL_DB_FILE, 'r') as f:
            db = json.load(f)
        print("Loaded existing database from local file.")
    else:
        db = get_playlist_videos()
        with open(LOCAL_DB_FILE, 'w') as f:
            json.dump(db, f, indent=4)
            
    for video in db:
        if video['is_processed']:
            continue
            
        print(f"\n⏳ Processing: {video['title']}")
        
        transcript = get_transcript(video['video_id'])
        if not transcript:
            print(f"⏭️ Skipping (No Subtitles).")
            # Marking as true so it doesn't get stuck in a loop later
            video['is_processed'] = True 
            video['ai_data'] = {"suggested_name": video['title'], "markdown_notes": "Notes unavailable.", "practice_test": []}
        else:
            ai_data = generate_ai_content(transcript, video['title'])
            if ai_data:
                video['ai_data'] = ai_data
                video['is_processed'] = True
                print(f"✅ Success! Clean Name: {ai_data.get('suggested_name')}")
            else:
                print(f"❌ Gemini generation failed. Retrying later.")
                continue 
                
        # Save progress
        with open(LOCAL_DB_FILE, 'w') as f:
            json.dump(db, f, indent=4)
            
        time.sleep(4) 
        
    print("\n🎉 All Videos Processed! Exporting to Vercel format...")
    export_json = {"course_notes": {"py_notes_shared": {}}}
    
    for v in db:
        if v.get('ai_data'):
            export_json["course_notes"]["py_notes_shared"][v['topic_id']] = v['ai_data']
            
    with open(FINAL_OUTPUT_FILE, 'w') as f:
        json.dump(export_json, f, separators=(',', ':')) 
        
    print(f"🚀 DONE! You can now upload '{FINAL_OUTPUT_FILE}' to Vercel.")

if __name__ == "__main__":
    run_local_engine()

In [0]:
import json
import time
from google import genai
from youtube_transcript_api import YouTubeTranscriptApi

# 1. API Setup
GEMINI_API_KEY = dbutils.secrets.get(scope="courseify", key="gemini-api")
client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Robust Transcript Function (🔥 UPDATED TO LATEST 2026 SYNTAX 🔥)
def get_transcript(video_id):
    try:
        # Patha 'get_transcript' badulu kotha 'fetch' method vadutunnam
        api = YouTubeTranscriptApi()
        transcript_list = api.fetch(video_id, languages=['en', 'en-IN', 'hi'])
        return " ".join([item['text'] for item in transcript_list])
    except Exception as e:
        print(f"Transcript Error for {video_id}: {e}")
        return None

# 3. Gemini Content Generator (THE ULTIMATE SUPER-PROMPT)
def generate_ai_content(transcript, title):
    prompt = f"""
    You are an Elite Silicon Valley Coding Instructor and a Master UX Content Designer for a premium educational app.
    Your objective is to transform the raw video transcript into a world-class, highly engaging, 'Topper-Level' study guide and adaptive practice test.
    
    Topic Context: '{title}'
    
    OUTPUT FORMAT: 
    Return EXACTLY a valid JSON object. Do not include ```json blocks or any plain text outside the JSON.
    
    JSON STRUCTURE & STRICT REQUIREMENTS:
    
    1. "suggested_name" (String): 
       - A clean, concise, professional title. 
       - Remove all YouTube fluff, channel names, pipe characters (|), and lecture numbers. 
       - Example: "History of Python | Python Tutorials for Beginners #lec2" -> "History of Python".
       
    2. "markdown_notes" (String):
       - Create visually stunning, easy-to-scan, and highly educational markdown text.
       - Structure it perfectly using this flow:
         * 🎯 **Overview:** A crisp 2-line summary of what this topic covers.
         * 📌 **Core Concepts:** Use clear sub-headings (###) and bullet points. Bold important keywords.
         * 💻 **Code in Action:** Provide 1-2 highly practical Python code examples. You MUST include detailed, easy-to-understand inline comments (`#`) explaining the logic step-by-step.
         * 💡 **Pro-Tip / ⚠️ Common Pitfall:** Add a small practical tip or a warning about a common mistake students make.
       - EXERCISE RULE: If the transcript is a "Coding Exercise", ignore the standard flow and structure the notes as: "Problem Statement" -> "Logic & Approach" -> "Solution Code" -> "Code Walkthrough".
       
    3. "practice_test" (Array of Objects):
       - Provide EXACTLY 3 multiple-choice questions.
       - ADAPTIVE RULE: 
         - If theory-based: Ask conceptual questions to test understanding.
         - If code/exercise-based: Ask logic-based questions or "Predict the Output" questions (using small code snippets in the question text).
       - Object Format: {{"q": "Question text?", "opts": ["Option 1", "Option 2", "Option 3", "Option 4"], "ans": 0, "exp": "One-line clear explanation why this is correct."}}
       
    Transcript to process: 
    {transcript[:8000]}
    """
    
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config={'temperature': 0.2}
        )
        res_text = response.text.replace('```json\n', '').replace('```', '').strip()
        return json.loads(res_text)
    except Exception as e:
        print(f"Gemini generation failed: {e}")
        return None

# 4. Main Processing Loop (BATCH MODE)
def process_pending_videos():
    # Only pick videos that haven't been processed AND haven't failed transcript extraction
    query = """
        SELECT video_links, topic 
        FROM courseify.default.courseify_data 
        WHERE notes = false AND (description IS NULL OR description != 'TRANSCRIPT_FAILED')
        LIMIT 5
    """
    pending_videos = spark.sql(query).collect()
    
    if not pending_videos:
        print("✅ All videos are fully processed or checked!")
        return
        
    print(f"Processing a batch of {len(pending_videos)} videos...\n")
    
    for row in pending_videos:
        v_id = row['video_links']
        v_title = row['topic']
        print(f"⏳ Processing: {v_title}")
        
        # Step A: Get Transcript (Using new Fetch logic)
        transcript = get_transcript(v_id)
        if not transcript:
            print(f"⏭️ Skipping (Transcript Unavailable). Marking as Failed in DB.")
            spark.sql(f"UPDATE courseify.default.courseify_data SET description = 'TRANSCRIPT_FAILED' WHERE video_links = '{v_id}'")
            continue
            
        # Step B: Generate Notes, QA & Clean Name
        ai_data = generate_ai_content(transcript, v_title)
        
        if ai_data:
            escaped_notes = ai_data.get('markdown_notes', '').replace("'", "\\'")
            escaped_qa = json.dumps(ai_data.get('practice_test', [])).replace("'", "\\'")
            escaped_desc = transcript[:1000].replace("'", "\\'") 
            
            clean_name = ai_data.get('suggested_name', v_title)
            escaped_name = clean_name.replace("'", "\\'")
            
            # Step C: Update the database
            update_query = f"""
                UPDATE courseify.default.courseify_data 
                SET description = '{escaped_desc}', 
                    notes_data = '{escaped_notes}', 
                    qa_data = '{escaped_qa}', 
                    suggested_name = '{escaped_name}',
                    notes = true, 
                    qa = true
                WHERE video_links = '{v_id}'
            """
            spark.sql(update_query)
            print(f"✅ Success! Cleaned Name: '{clean_name}'")
        else:
            print(f"❌ Failed AI Generation for: {v_title}")
            
        # SLEEP to respect API limits
        time.sleep(5) 

    print("\nBatch Complete! Run this cell again to process the next 5 videos.")

# Run the engine
process_pending_videos()

In [0]:
import json
import time
from google import genai
from youtube_transcript_api import YouTubeTranscriptApi

# 1. API Setup
GEMINI_API_KEY = dbutils.secrets.get(scope="courseify", key="gemini-api")
client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Robust Transcript Function
def get_transcript(video_id):
    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=['en', 'en-IN', 'hi'])
        return " ".join([item['text'] for item in transcript_list])
    except Exception as e:
        print(f"Transcript Error for {video_id}: {e}")
        return None

# 3. Gemini Content Generator (THE ULTIMATE SUPER-PROMPT)
def generate_ai_content(transcript, title):
    prompt = f"""
    You are an Elite Silicon Valley Coding Instructor and a Master UX Content Designer for a premium educational app.
    Your objective is to transform the raw video transcript into a world-class, highly engaging, 'Topper-Level' study guide and adaptive practice test.
    
    Topic Context: '{title}'
    
    OUTPUT FORMAT: 
    Return EXACTLY a valid JSON object. Do not include ```json blocks or any plain text outside the JSON.
    
    JSON STRUCTURE & STRICT REQUIREMENTS:
    
    1. "suggested_name" (String): 
       - A clean, concise, professional title. 
       - Remove all YouTube fluff, channel names, pipe characters (|), and lecture numbers. 
       - Example: "History of Python | Python Tutorials for Beginners #lec2" -> "History of Python".
       
    2. "markdown_notes" (String):
       - Create visually stunning, easy-to-scan, and highly educational markdown text.
       - Structure it perfectly using this flow:
         * 🎯 **Overview:** A crisp 2-line summary of what this topic covers.
         * 📌 **Core Concepts:** Use clear sub-headings (###) and bullet points. Bold important keywords.
         * 💻 **Code in Action:** Provide 1-2 highly practical Python code examples. You MUST include detailed, easy-to-understand inline comments (`#`) explaining the logic step-by-step.
         * 💡 **Pro-Tip / ⚠️ Common Pitfall:** Add a small practical tip or a warning about a common mistake students make.
       - EXERCISE RULE: If the transcript is a "Coding Exercise", ignore the standard flow and structure the notes as: "Problem Statement" -> "Logic & Approach" -> "Solution Code" -> "Code Walkthrough".
       
    3. "practice_test" (Array of Objects):
       - Provide EXACTLY 3 multiple-choice questions.
       - ADAPTIVE RULE: 
         - If theory-based: Ask conceptual questions to test understanding.
         - If code/exercise-based: Ask logic-based questions or "Predict the Output" questions (using small code snippets in the question text).
       - Object Format: {{"q": "Question text?", "opts": ["Option 1", "Option 2", "Option 3", "Option 4"], "ans": 0, "exp": "One-line clear explanation why this is correct."}}
       
    Transcript to process: 
    {transcript[:8000]}
    """
    
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config={'temperature': 0.2}
        )
        res_text = response.text.replace('```json\n', '').replace('```', '').strip()
        return json.loads(res_text)
    except Exception as e:
        print(f"Gemini generation failed: {e}")
        return None

# 4. Main Processing Loop (BATCH MODE)
def process_pending_videos():
    # FIX: Fetching 'topic' (Video Title). Also ignoring previously failed transcripts so we don't get stuck.
    query = """
        SELECT video_links, topic 
        FROM courseify.default.courseify_data 
        WHERE notes = false AND (description IS NULL OR description != 'TRANSCRIPT_FAILED')
        LIMIT 5
    """
    pending_videos = spark.sql(query).collect()
    
    if not pending_videos:
        print("✅ All videos are fully processed or checked!")
        return
        
    print(f"Processing a batch of {len(pending_videos)} videos...\n")
    
    for row in pending_videos:
        v_id = row['video_links']
        v_title = row['topic'] # FIX: Actual Video Title
        print(f"⏳ Processing: {v_title}")
        
        # Step A: Get Transcript
        transcript = get_transcript(v_id)
        if not transcript:
            print(f"⏭️ Skipping (Transcript Unavailable). Marking as Failed in DB.")
            spark.sql(f"UPDATE courseify.default.courseify_data SET description = 'TRANSCRIPT_FAILED' WHERE video_links = '{v_id}'")
            continue
            
        # Step B: Generate Notes, QA & Clean Name
        ai_data = generate_ai_content(transcript, v_title)
        
        if ai_data:
            escaped_notes = ai_data.get('markdown_notes', '').replace("'", "\\'")
            escaped_qa = json.dumps(ai_data.get('practice_test', [])).replace("'", "\\'")
            escaped_desc = transcript[:1000].replace("'", "\\'") 
            
            clean_name = ai_data.get('suggested_name', v_title)
            escaped_name = clean_name.replace("'", "\\'")
            
            # Step C: Update the database (Only on true success)
            update_query = f"""
                UPDATE courseify.default.courseify_data 
                SET description = '{escaped_desc}', 
                    notes_data = '{escaped_notes}', 
                    qa_data = '{escaped_qa}', 
                    suggested_name = '{escaped_name}',
                    notes = true, 
                    qa = true
                WHERE video_links = '{v_id}'
            """
            spark.sql(update_query)
            print(f"✅ Success! Cleaned Name: '{clean_name}'")
        else:
            print(f"❌ Failed AI Generation for: {v_title}")
            
        # SLEEP to respect API limits
        time.sleep(5) 

    print("\nBatch Complete! Run this cell again to process the next 5 videos.")

# Run the engine
process_pending_videos()

In [0]:
# import json
# import time
# from google import genai
# from youtube_transcript_api import YouTubeTranscriptApi

# # 1. API Setup (Using the new SDK)
# GEMINI_API_KEY = dbutils.secrets.get(scope="courseify", key="gemini-api")
# client = genai.Client(api_key=GEMINI_API_KEY)

# # 2. Get Transcript Function
# def get_transcript(video_id):
#     try:
#         # YouTube auto-generates subtitles, fetching them here
#         transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=['en', 'en-IN', 'hi'])
#         return " ".join([item['text'] for item in transcript_list])
#     except Exception as e:
#         print(f"Transcript failed for {video_id}: {e}")
#         return None

# # # 3. Gemini Content Generator
# # def generate_ai_content(transcript, title):
# #     prompt = f"""
# #     You are an expert coding tutor. Based ONLY on the following video transcript for the topic '{title}', generate study notes, a practice test, and a clean suggested topic name.
    
# #     REQUIREMENTS:
# #     - "suggested_name": A clean, concise, professional title for this topic. Remove all YouTube-specific fluff, channel names, pipe characters (|), and lecture numbers (like #lec1). Example: "History of Python | Python Tutorials for Beginners #lec2" should become just "History of Python".
# #     - "markdown_notes": Beautiful markdown text (headings, bullet points, 1-2 Python code examples).
# #     - "practice_test": Array of exactly 2 multiple choice questions based on the transcript.
    
# #     Return EXACTLY a valid JSON object with keys: "suggested_name", "markdown_notes", and "practice_test".
# #     Format for practice test objects: {{"q": "...", "opts": ["A", "B", "C", "D"], "ans": 0, "exp": "..."}}
    
# #     Transcript: {transcript[:8000]}
# #     """
    
# #     try:
# #         response = client.models.generate_content(
# #             model='gemini-2.5-flash',
# #             contents=prompt
# #         )
# #         # Clean the response to parse JSON safely
# #         res_text = response.text.replace('```json\n', '').replace('```', '').strip()
# #         return json.loads(res_text)
# #     except Exception as e:
# #         print(f"Gemini generation failed: {e}")
# #         return None

# # 3. Gemini Content Generator (THE ULTIMATE SUPER-PROMPT)
# def generate_ai_content(transcript, title):
#     prompt = f"""
#     You are an Elite Silicon Valley Coding Instructor and a Master UX Content Designer for a premium educational app.
#     Your objective is to transform the raw video transcript into a world-class, highly engaging, 'Topper-Level' study guide and adaptive practice test.
    
#     Topic Context: '{title}'
    
#     OUTPUT FORMAT: 
#     Return EXACTLY a valid JSON object. Do not include ```json blocks or any plain text outside the JSON.
    
#     JSON STRUCTURE & STRICT REQUIREMENTS:
    
#     1. "suggested_name" (String): 
#        - A clean, concise, professional title. 
#        - Remove all YouTube fluff, channel names, pipe characters (|), and lecture numbers. 
#        - Example: "History of Python | Python Tutorials for Beginners #lec2" -> "History of Python".
       
#     2. "markdown_notes" (String):
#        - Create visually stunning, easy-to-scan, and highly educational markdown text.
#        - Structure it perfectly using this flow:
#          * 🎯 **Overview:** A crisp 2-line summary of what this topic covers.
#          * 📌 **Core Concepts:** Use clear sub-headings (###) and bullet points. Bold important keywords.
#          * 💻 **Code in Action:** Provide 1-2 highly practical Python code examples. You MUST include detailed, easy-to-understand inline comments (`#`) explaining the logic step-by-step.
#          * 💡 **Pro-Tip / ⚠️ Common Pitfall:** Add a small practical tip or a warning about a common mistake students make.
#        - EXERCISE RULE: If the transcript is a "Coding Exercise", ignore the standard flow and structure the notes as: "Problem Statement" -> "Logic & Approach" -> "Solution Code" -> "Code Walkthrough".
       
#     3. "practice_test" (Array of Objects):
#        - Provide EXACTLY more than 5 multiple-choice questions.
#        - ADAPTIVE RULE: 
#          - If theory-based: Ask conceptual questions to test understanding.
#          - If code/exercise-based: Ask logic-based questions or "Predict the Output" questions (using small code snippets in the question text).
#        - Object Format: {{"q": "Question text?", "opts": ["Option 1", "Option 2", "Option 3", "Option 4"], "ans": 0, "exp": "One-line clear explanation why this is correct."}}
       
#     Transcript to process: 
#     {transcript[:8000]}
#     """
    
#     try:
#         response = client.models.generate_content(
#             model='gemini-2.5-flash',
#             contents=prompt,
#             config={'temperature': 0.2} # Highly focused and deterministic output
#         )
#         res_text = response.text.replace('```json\n', '').replace('```', '').strip()
#         return json.loads(res_text)
#     except Exception as e:
#         print(f"Gemini generation failed: {e}")
#         return None

# # 4. Main Processing Loop (BATCH MODE: Process 5 videos at a time)
# def process_pending_videos():
#     # Fetch only 5 videos where notes are not yet generated
#     pending_videos = spark.sql("SELECT video_links, title FROM courseify.default.courseify_data WHERE notes = false LIMIT 5").collect()
    
#     if not pending_videos:
#         print("✅ All videos are fully processed with Notes, QA, and Clean Names!")
#         return
        
#     print(f"Processing a batch of {len(pending_videos)} videos...\n")
    
#     for row in pending_videos:
#         v_id = row['video_links']
#         v_title = row['title']
#         print(f"⏳ Processing: {v_title}")
        
#         # Step A: Get Transcript
#         transcript = get_transcript(v_id)
#         if not transcript:
#             # If no subtitles exist, mark as processed but empty
#             spark.sql(f"UPDATE courseify.default.courseify_data SET notes = true, qa = true, notes_data = 'Notes not available', qa_data = '[]', suggested_name = '{v_title}' WHERE video_links = '{v_id}'")
#             continue
            
#         # Step B: Generate Notes, QA & Clean Name
#         ai_data = generate_ai_content(transcript, v_title)
        
#         if ai_data:
#             # Escape single quotes for SQL insertion
#             escaped_notes = ai_data['markdown_notes'].replace("'", "\\'")
#             escaped_qa = json.dumps(ai_data['practice_test']).replace("'", "\\'")
#             escaped_desc = transcript[:1000].replace("'", "\\'") # Save a snippet of transcript
            
#             # Use suggested name, or fallback to original title if missing
#             clean_name = ai_data.get('suggested_name', v_title)
#             escaped_name = clean_name.replace("'", "\\'")
            
#             # Step C: Update the database
#             update_query = f"""
#                 UPDATE courseify.default.courseify_data 
#                 SET description = '{escaped_desc}', 
#                     notes_data = '{escaped_notes}', 
#                     qa_data = '{escaped_qa}', 
#                     suggested_name = '{escaped_name}',
#                     notes = true, 
#                     qa = true
#                 WHERE video_links = '{v_id}'
#             """
#             spark.sql(update_query)
#             print(f"✅ Success! Cleaned Name: '{clean_name}'")
#         else:
#             print(f"❌ Failed AI Generation for: {v_title}")
            
#         # 🚨 SLEEP for 5 seconds to avoid Gemini Free Tier API Rate Limits
#         time.sleep(5) 

#     print("\nBatch Complete! Run this cell again to process the next 5 videos.")

# # Run the engine
# process_pending_videos()